In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.metrics.pairwise import cosine_similarity



In [3]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

## Q1


In [4]:
answer_counts = df["answer"].value_counts()

print(answer_counts)

most_freq = answer_counts.max()
least_freq = answer_counts.min()

result_q1 = most_freq + least_freq

print("Most frequent:", most_freq)
print("Least frequent:", least_freq)
print("Answer:", result_q1)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most frequent: 490
Least frequent: 324
Answer: 814


## Q2

In [5]:
translator = str.maketrans('', '', string.punctuation)

clean_prompts = (
    df["prompt"]
    .astype(str)
    .str.lower()
    .apply(lambda x: x.translate(translator))
)

vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

result_q2 = len(vocab)

print("Vocabulary Size:", result_q2)

Vocabulary Size: 859


## Q3

In [6]:
translator = str.maketrans('', '', string.punctuation)

row = df[df["id"] == 1].iloc[0]

clean_prompt = (
    str(row["prompt"])
    .lower()
    .translate(translator)
)

tokens = clean_prompt.split()

filtered_tokens = [
    word
    for word in tokens
    if word not in ENGLISH_STOP_WORDS
]

result_q3 = len(filtered_tokens)

print("Words remaining:", result_q3)

Words remaining: 13


## Q4

In [7]:
combined_text = (
    df["prompt"].fillna("") + " " +
    df["A"].fillna("") + " " +
    df["B"].fillna("") + " " +
    df["C"].fillna("") + " " +
    df["D"].fillna("") + " " +
    df["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")

vectorizer.fit(combined_text)

result_q4 = len(vectorizer.vocabulary_)

print("Feature Columns:", result_q4)

Feature Columns: 2762


## Q5

In [8]:
row = df[df["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([str(row["prompt"])])
option_a_vec = vectorizer.transform([str(row["A"])])

similarity = cosine_similarity(
    prompt_vec,
    option_a_vec
)[0][0]

result_q5 = round(similarity, 4)

print("Similarity:", result_q5)

Similarity: 0.272


## Q6

In [9]:
correct = 0

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform(
        [str(row["prompt"])]
    )

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform(
            [str(row[option])]
        )

        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    predicted = max(
        similarities,
        key=similarities.get
    )

    if predicted == row["answer"]:
        correct += 1

accuracy = correct / len(df) * 100

print("Percentage:", round(accuracy, 2))

Percentage: 13.55


## Q7

In [10]:
actual = "C"
preds = ["C", "A", "B"]

score = 0

for rank, pred in enumerate(preds, start=1):
    if pred == actual:
        score = 1 / rank
        break

print(score)

1.0


## Q8

In [11]:
actual = "B"
preds = ["D", "B", "E"]

score = 0

for rank, pred in enumerate(preds, start=1):
    if pred == actual:
        score = 1 / rank
        break

print(score)

0.5


In [12]:
def map3_score(actual, preds):
    for rank, pred in enumerate(preds[:3], start=1):
        if pred == actual:
            return 1.0 / rank
    return 0.0

## Q9

In [13]:
answer_counts = df["answer"].value_counts()

top3 = answer_counts.index[:3].tolist()

print("Top 3 classes:", top3)

scores = []

for actual in df["answer"]:
    scores.append(
        map3_score(actual, top3)
    )

majority_map3 = np.mean(scores)

print("Majority Baseline MAP@3:", majority_map3)

Top 3 classes: ['B', 'C', 'A']
Majority Baseline MAP@3: 0.42125


## Q10

In [14]:
scores = []

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform(
        [str(row["prompt"])]
    )

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform(
            [str(row[option])]
        )

        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        similarities.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top3_predictions = [
        x[0]
        for x in ranked[:3]
    ]

    scores.append(
        map3_score(
            row["answer"],
            top3_predictions
        )
    )

tfidf_map3 = np.mean(scores)

print("TF-IDF MAP@3:", tfidf_map3)

TF-IDF MAP@3: 0.2961666666666667
